In [1]:
###CHANGELOGG!!!!!

# errorytpe > relative
# objective > multiobjective
# run_BUSBOI >  not bedthenQ, not solver

#version D

#######
single spline



In [1]:
suppressMessages({
    library(dplyr)
    library(parallel)
    library(ggplot2)
    library(tidyr)
    library(hydroGOF)
    })

In [2]:
gauge_df=readRDS('/nas/cee-water/cjgleason/colin/analyze confluence runs/SVS_df.rds')
gauged_reaches=unique(gauge_df$reach_id)
# swot_base='/nas/cee-water/cjgleason/ellie/SWOT/confluence/confluence_relPermML/relPermML_mnt/input/swot/'
# sos_base='/nas/cee-water/cjgleason/ellie/SWOT/confluence/confluence_relPermML/relPermML_mnt/input/sos/'
swot_base='/nas/cee-ice/data/Confluence_Runs/global_vD/global_vD_mnt/input/swot/'
sos_base='/nas/cee-ice/data/Confluence_Runs/global_vD/global_vD_mnt/input/sos/'

reach_ids=gauged_reaches[gauged_reaches %in% substr(list.files(swot_base),1,11)]

In [3]:
output_path='/nas/cee-water/cjgleason/colin/BUSBOI/debug tests/daily_single_splineD/'
run_ids=substr(list.files(output_path),1,11)
unrun= reach_ids[!reach_ids %in% run_ids]
length(unrun)

[1] 31

In [15]:
# #save a test case for AWS testing
# #needs:
#     #SWORD
#     #Priors
#     #SWOT input data
# this_reach_id='56424600151'
# sword='/nas/cee-ice/data/SWORD/SWORDv16/netcdf/oc_sword_v16.nc'
# priors=paste0(sos_base,'oc_sword_v16_SOS_priors.nc')
# swot=paste0(swot_base,this_reach_id,'_SWOT.nc')

In [5]:
# versionD=as.data.frame(readRDS('/nas/cee-water/cjgleason/colin/SWOT_global_Q_paper/SES_dataframe_versionD.rds'))%>%
#     select(-geometry)
# versionC=as.data.frame(readRDS('/nas/cee-water/cjgleason/colin/SWOT_global_Q_paper/SES_dataframe.rds'))%>%
#     select(-geometry)

In [5]:
# nrow(filter(versionD,model=='ML_ensemble'))
# nrow(filter(versionC,model=='ML_ensemble'))

In [7]:

# readRDS(list.files('/nas/cee-water/cjgleason/colin/SWOT_global_Q_paper/daily_ensembles_versionD/',full.names=TRUE)[31234])

In [7]:
source('/nas/cee-water/cjgleason/colin/BUSBOI/BUSBOI/main_function.R')
suppressWarnings({
for (i in 19:26){

    unrun=reach_ids
    print(i)
    print(unrun[i])

    
    
test= main_function(unrun[i],
                   output_path=output_path,
                   swot_base=swot_base,
                   sos_base=sos_base,
                   Q_prior='daily', #'daily' or 'monthly'
                   tulip='OFF', #'ON' or 'OFF'
                   GVF_on=0, # 0 or 1
                   fix_bed=0) # 0 = 5 pt, 1 = 1pt, 2 = fixed
    
}

})

# a= Sys.time()
# clust=makeCluster(40)
# test=parLapply(clust,unrun[1:40],main_function,
#                    output_path=output_path,
#                    swot_base=swot_base,
#                    sos_base=sos_base,
#                    Q_prior='daily', #or 'monthly'
#                    tulip='OFF', #or 'OFF'
#                    GVF_on=0,
#                    fix_bed=0) # 0 = 5 pt, 1 = 1pt, 2 = fixed
# stopCluster(clust)
# print(Sys.time()-a)

[1] 19
[1] "78100600061"


ERROR: Error in FUN(newX[, i], ...): object 'bonk' not found


In [10]:
stopCluster(clust)

In [46]:
###to control where the slurm files are written
working_dir='/nas/cee-water/cjgleason/colin/BUSBOI/debug_logs/'
setwd(working_dir)

source('/nas/cee-water/cjgleason/colin/BUSBOI/BUSBOI/main_function.R')

library(rslurm, lib.loc = "/nas/cee-water/cjgleason/r-lib/",quietly = TRUE)
library(whisker, lib.loc = "/nas/cee-water/cjgleason/r-lib/",quietly = TRUE)

testname='daily'
#slurm block
slurm_options= list(mem=64000, 'time'='50:00:00', #options for memory, time, parition, and an error file
                    partition ='ceewater_cjgleason-cpu',
                    error='slurm-%A_%a.err')
                    # nodelist = 'ceewater-cpu008')
                    # exclude='ceewater-cpu009')
sjob <- slurm_map(as.list(unrun), #thing you want to loop over. must be a list
                  main_function,  # name of the function. declared above with the 'source' command
                  jobname = testname, # defined above. just for convenience
                   output_path=output_path,
                   swot_base=swot_base,
                   sos_base=sos_base,
                  Q_prior='daily',
                  tulip='OFF',
                  GVF_on=0,
                  fix_bed=0,  #0 = 5pts, 1= 1pt, 2 = fixed
                  nodes = 1, # many nodes do you want?
                  preschedule_cores=FALSE, # keep this FALSE
                  cpus_per_node = 30, # how many CPUS per node. 
                  submit = TRUE, # if TRUE, submits to the cluster
                  slurm_options=slurm_options, #defined above
                  libPaths="/nas/cee-water/cjgleason/r-lib/" ) #library paths

Submitted batch job 53589424



In [23]:
stopCluster(clust)